# Chapter 8 — Learning Signal and Ignoring Noise: Introduction to Regularization and Batching

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Deep Learning

## Tujuan bab
Bab ini membahas **overfitting**, **dropout**, dan **batch gradient descent**. Contoh utama buku menggunakan klasifikasi digit MNIST dengan jaringan tiga lapisan.

## Ringkasan teori dari buku
- **Overfitting:** model mempelajari detail spesifik pada data pelatihan sehingga kinerjanya menurun pada data baru. Akurasi training yang tinggi saja bukan bukti model dapat melakukan generalisasi.
- **Generalization:** kemampuan menghasilkan prediksi yang baik pada data yang tidak digunakan ketika melatih model.
- **Regularisasi:** teknik untuk mengurangi kecenderungan model menghafal data pelatihan.
- **Dropout:** selama pelatihan, sebagian aktivasi dinonaktifkan secara acak. Ini mencegah jaringan terlalu bergantung pada jalur/neuron tertentu.
- **Batch gradient descent:** pembaruan bobot dihitung berdasarkan sekumpulan contoh (batch), bukan selalu satu contoh per pembaruan. Ini dapat membuat perhitungan lebih efisien dan gradien lebih stabil.

Notebook ini menggunakan contoh numerik sintetis agar dapat dijalankan tanpa mengunduh MNIST atau memasang Keras/TensorFlow. Contoh tersebut mendemonstrasikan konsep, bukan mereproduksi eksperimen MNIST buku secara persis.

In [ ]:
import numpy as np

# Data sintetis: target sebenarnya bergantung pada pola sederhana.
rng = np.random.default_rng(7)
X = rng.normal(size=(240, 2))
y = ((X[:, 0] + 0.7 * X[:, 1]) > 0).astype(int)

# Bagi train/test secara deterministik.
X_train, y_train = X[:180], y[:180]
X_test, y_test = X[180:], y[180:]

print("Ukuran train:", X_train.shape, y_train.shape)
print("Ukuran test :", X_test.shape, y_test.shape)

## 1. Mengukur generalisasi
Akurasi training dan test harus dilihat bersama. Perbedaan yang besar di antara keduanya dapat menjadi indikasi overfitting. Pada eksperimen nyata, pembagian data sebaiknya dilakukan secara acak dan berstrata bila diperlukan.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -40, 40)))

def train_logistic(X, y, lr=0.2, epochs=250, batch_size=None, l2=0.0, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1])
    b = 0.0
    n = len(X)
    bs = n if batch_size is None else batch_size

    for _ in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, bs):
            idx = order[start:start + bs]
            xb, yb = X[idx], y[idx]
            p = sigmoid(xb @ w + b)
            dw = xb.T @ (p - yb) / len(idx) + l2 * w
            db = np.mean(p - yb)
            w -= lr * dw
            b -= lr * db
    return w, b

def accuracy(X, y, w, b):
    return np.mean((sigmoid(X @ w + b) >= 0.5) == y)

w_sgd, b_sgd = train_logistic(X_train, y_train, batch_size=1, seed=1)
w_batch, b_batch = train_logistic(X_train, y_train, batch_size=32, seed=1)
w_reg, b_reg = train_logistic(X_train, y_train, batch_size=32, l2=0.05, seed=1)

for name, w, b in [
    ("SGD (batch_size=1)", w_sgd, b_sgd),
    ("Mini-batch (32)", w_batch, b_batch),
    ("Mini-batch + L2", w_reg, b_reg),
]:
    print(f"{name:22s} train={accuracy(X_train, y_train, w, b):.3f} test={accuracy(X_test, y_test, w, b):.3f}")

## 2. Ilustrasi dropout
Dropout mematikan sebagian aktivasi secara acak **selama training**. Berikut hanya demonstrasi mask dropout; ini bukan implementasi lengkap pelatihan jaringan dengan dropout.

In [ ]:
activations = np.array([0.8, 0.1, 0.6, 0.9, 0.4])
keep_probability = 0.6
rng = np.random.default_rng(3)
mask = (rng.random(activations.shape) < keep_probability).astype(float)

# Inverted dropout: aktivasi yang dipertahankan diskalakan agar nilai harapan tetap.
dropped = activations * mask / keep_probability

print("Aktivasi awal :", activations)
print("Mask dropout  :", mask)
print("Setelah dropout:", dropped)

## Analisis
- Evaluasi pada data test penting karena model dapat menghafal data training.
- Mini-batch mengubah frekuensi dan sifat pembaruan parameter.
- Regularisasi menambahkan kendala/penalti agar parameter tidak tumbuh tanpa kebutuhan.
- Dropout mengurangi ketergantungan pada aktivasi tertentu saat training.

## Kesimpulan
Bab 8 menggeser perhatian dari sekadar menurunkan error training menuju pembelajaran pola yang tetap berguna pada data baru. Bandingkan training dan test, dan gunakan teknik regularisasi sesuai masalah.